In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import random


In [2]:
def generate_polynomial_data(n_points=100, noise_std=0.2, seed=42):
    """
    Generates synthetic (x, y) data for a polynomial y = 3x^3 + 2x^2 - 4x + 1 with added Gaussian noise.
    
    Args:
        n_points (int): Number of data points to generate.
        noise_std (float): Standard deviation of the Gaussian noise.
        seed (int): Random seed for reproducibility.

    Returns:
        (x, y): Tuple of Tensors representing inputs and polynomial outputs.
    """
    # Set random seed for reproducibility
    torch.manual_seed(seed)
    
    # Generate equally spaced x values
    x = torch.linspace(-3, 3, n_points).unsqueeze(1)  # shape: (n_points, 1)

    # Define polynomial: 3x^3 + 2x^2 - 4x + 1
    y_true = 3 * x**3 + 2 * x**2 - 4 * x + 1
    
    # Add Gaussian noise
    noise = torch.randn_like(y_true) * noise_std
    
    # Final y = polynomial + noise
    y = y_true + noise
    
    return x, y


In [3]:
x, y = generate_polynomial_data(n_points=1000, noise_std=0.5)

In [4]:
def generate_polynomial_data_new(n_points=100, noise_std=0.2, seed=42):
    """
    Generates synthetic (x, y) data for a polynomial y = 3x^3 + 2x^2 - 4x + 2 
    with added Gaussian noise.
    
    Args:
        n_points (int): Number of data points to generate.
        noise_std (float): Standard deviation of the Gaussian noise.
        seed (int): Random seed for reproducibility.

    Returns:
        (x, y): Tuple of Tensors representing inputs and polynomial outputs.
    """
    # Set random seed for reproducibility
    torch.manual_seed(seed)
    
    # Generate equally spaced x values
    x = torch.linspace(-3, 3, n_points).unsqueeze(1)  # shape: (n_points, 1)

    # Define polynomial: 3x^3 + 2x^2 - 4x + 2
    y_true = 3 * x**3 + 2 * x**2 - 4 * x + 2
    
    # Add Gaussian noise
    noise = torch.randn_like(y_true) * noise_std
    
    # Final y = polynomial + noise
    y = y_true + noise
    
    return x, y

In [5]:
x_new, y_new = generate_polynomial_data_new(n_points=1000, noise_std=0.75)

In [6]:
class ThreeLayerNet(nn.Module):
    def __init__(self):
        super(ThreeLayerNet, self).__init__()
        # Define a simple feed-forward network:
        # Input -> Hidden1 -> Hidden2 -> Output
        self.net = nn.Sequential(
            nn.Linear(1, 16),   # Input is 1-D -> 16 neurons
            nn.ReLU(),
            nn.Linear(16, 16),  # 16 neurons -> 16 neurons
            nn.ReLU(),
            nn.Linear(16, 1)    # 16 neurons -> 1-D output
        )
    
    def forward(self, x):
        return self.net(x)

In [7]:
model = ThreeLayerNet()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.02)

In [8]:
# Training loop
for epoch in range(100):
    optimizer.zero_grad()
    y_pred = model(x)

    loss = criterion(y_pred, y)

    loss.backward()
    optimizer.step()
    #print(f'loss: {loss.item()}')

Online learning

In [9]:
y_pred = model(x)
loss = criterion(y_pred, y)
print(f'loss: {loss.item()}')

loss: 19.194278717041016


In [10]:
y_pred = model(x_new)
loss = criterion(y_pred, y_new)
print(f'loss: {loss.item()}')

loss: 20.4167423248291


In [11]:
class OModelAutograd(nn.Module):
    def __init__(self, model: nn.Module, params: list, criterion=nn.MSELoss(), lr=0.001):
        """
        Initialize the optimizer-like module.

        model: The model to optimize (used for forward computation).
        params: A list of tensors representing the parameters to be updated.
        criterion: Loss function (default is MSELoss).
        lr: Learning rate for manual updates.
        """
        super(OModelAutograd, self).__init__()
        self.model = model  # Store the model for the forward pass
        self.params = [p for p in params if p.requires_grad]  # Filter parameters requiring gradients
        self.criterion = criterion
        self.lr = torch.tensor(lr, dtype=torch.float32)  # Convert learning rate to Tensor

    def forward_fn(self, x):
        """
        Perform the forward pass using the provided model.
        """
        return self.model(x)

    def forward(self, x: torch.Tensor, x_prev: torch.Tensor, y_prev: torch.Tensor) -> torch.Tensor:
        """
        Perform a forward pass, compute the loss, manually update the parameters,
        and return the updated prediction.

        x: Input tensor
        y_prev: Target tensor
        """
        # 1. Forward pass
        pred = self.forward_fn(x_prev)  # Call the internal forward function
        loss = self.criterion(pred, y_prev)

        # 2. Compute gradients for the given parameters
        grads = torch.autograd.grad(
            [loss], self.params, create_graph=False, retain_graph=False, allow_unused=True
        )

        # 3. Manual gradient descent step
        with torch.no_grad():
            for param, grad in zip(self.params, grads):
                if grad is not None:  # Handle None gradients
                    param -= self.lr * grad

        # 4. Return the updated prediction
        return self.forward_fn(x)


In [12]:
new_model = OModelAutograd(
    model = model,
    params=list(model.parameters()),  # Pass the model's parameters explicitly
    criterion=nn.MSELoss(),  # Specify the loss function
    lr=0.00001  # Learning rate
)

# Example training loop
for _ in range(200):
    xp = torch.zeros(size=(1,))
    yp = torch.zeros(size=(1,))

    for xi,yi in zip(x_new,y_new):
        y_pred = new_model(xi, xp, yp)  # Forward pass, loss calculation, and parameter update
        loss = new_model.criterion(y_pred, y)  # Optional: track the loss

        # Do swap
        xp = xi
        yp = yi

    # Eval
    xp = torch.zeros_like(x_new)
    yp = torch.zeros_like(y_new)
    y_pred = new_model(x_new, xp, yp)
    loss = criterion(y_pred, y_new) 
    print(f'Loss: {loss.item()}')

    # Shuffle x,y
    # Assume x_new and y_new are lists or tensors
    data = list(zip(x_new, y_new))  # Combine into pairs
    random.shuffle(data)  # Shuffle the pairs
    x_new, y_new = zip(*data)  # Unzip back into separate variables

    # If x_new and y_new are tensors, convert them back
    x_new = torch.stack(x_new)
    y_new = torch.stack(y_new)




/home/ella/.my_conda/envs/climate/lib/python3.12/site-packages/torch/nn/modules/loss.py:608: UserWarning: Using a target size (torch.Size([1000, 1])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Loss: 62.733219146728516
Loss: 18.291845321655273
Loss: 14.220511436462402
Loss: 13.188493728637695
Loss: 11.482172966003418
Loss: 9.983599662780762
Loss: 9.192410469055176
Loss: 8.519338607788086
Loss: 7.175337314605713
Loss: 6.606476306915283
Loss: 6.035116672515869
Loss: 5.575895309448242
Loss: 4.978187561035156
Loss: 5.0027689933776855
Loss: 4.329914569854736
Loss: 4.345986366271973
Loss: 4.068696022033691
Loss: 3.6132185459136963
Loss: 3.838989019393921
Loss: 3.343520164489746
Loss: 3.3614628314971924
Loss: 3.7703914642333984
Loss: 2.9513354301452637
Loss: 2.787137985229492
Loss: 2.7122642993927
Loss: 2.8230090141296387
Loss: 2.5461692810058594
Loss: 2.4307711124420166
Loss: 2.517178773880005
Loss: 2.275831699371338
Loss: 2.351907968521118
Loss: 2.16758131980896
Loss: 2.1052427291870117
Loss: 2.135199785232544
Loss: 2.042236089706421
Loss: 1.9392735958099365
Loss: 1.94136643409729
Loss: 1.843624472618103
Loss: 1.8254231214523315
Loss: 1.7807366847991943
Loss: 1.7376599311828613
Lo

Save as script

In [13]:
script_model = torch.jit.script(new_model)
script_model.save(f"./online_example.pt")
